# Búsqueda de hiperparámetros (Optuna) para YOLOv8n

Notebook reproducible en **Google Colab** y también ejecutable en local.

Flujo:
1. Setup e imports
2. Carga del dataset
3. Definición del objetivo de Optuna (con pruning)
4. Ejecución de la búsqueda (por defecto 10 trials x 10 épocas)
5. Visualización en TensorBoard
6. Visualización de resultados de optimización (Optuna)
7. Carga del mejor `best.pt` y revisión de curvas/matriz
8. Inferencia con `kortxo.jpg` usando el mejor modelo

In [1]:
# Fase 0 (setup): instalación de dependencias + imports globales
# Breve: dejamos el entorno listo para entrenar YOLOv8 con Optuna + TensorBoard.

import sys

if 'google.colab' in sys.modules:
    !pip -q install ultralytics optuna tensorboard huggingface_hub pyyaml seaborn
else:
    !pip -q install ultralytics optuna tensorboard huggingface_hub pyyaml seaborn

from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
import yaml
from huggingface_hub import snapshot_download
from IPython.display import Image, display
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
from ultralytics import YOLO

sns.set_theme(style='whitegrid')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Carga del dataset
Breve: descargamos el dataset desde Hugging Face y localizamos su archivo `data.yml`.

In [2]:
DATASET_REPO = 'mikeldiez/kortxovision'
LOCAL_DATA_DIR = Path('./data/kortxovision').resolve()
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Descargando dataset {DATASET_REPO}...')
dataset_root = Path(
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type='dataset',
        local_dir=str(LOCAL_DATA_DIR),
        local_dir_use_symlinks=False,
    )
).resolve()
print('Dataset descargado en:', dataset_root)

candidate_yaml = (
    list(dataset_root.rglob('data.yml'))
    + list(dataset_root.rglob('data.yaml'))
    + list(dataset_root.rglob('dataset.yml'))
    + list(dataset_root.rglob('dataset.yaml'))
)

if not candidate_yaml:
    raise FileNotFoundError('No se encontró data.yml/data.yaml en el dataset descargado.')

data_yaml_path = candidate_yaml[0]
print('Usando YAML:', data_yaml_path)

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print('\nResumen data.yaml:')
print({'nc': data_cfg.get('nc'), 'train': data_cfg.get('train'), 'val': data_cfg.get('val')})

Descargando dataset mikeldiez/kortxovision...


/home/mikel/github/TKNIKA/kortxovision/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching ... files: 2102it [00:02, 803.72it/s]


Dataset descargado en: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision
Usando YAML: /home/mikel/github/TKNIKA/kortxovision/notebooks/data/kortxovision/data.yml

Resumen data.yaml:
{'nc': 17, 'train': 'images/train', 'val': 'images/val'}


## 2) Definir búsqueda de hiperparámetros
Breve: configuramos Optuna para maximizar `mAP50-95` de validación en entrenamientos cortos.

In [ ]:
# Configuración principal (editable para clase/demo)
MODEL_NAME = 'yolov8n.pt'
N_TRIALS = 10            # Número de experimentos (configurable)
EPOCHS_PER_TRIAL = 10    # Épocas por experimento (configurable)

IMG_SIZE = 640
BATCH_SIZE = 8
WORKERS = 2
DEVICE = None            # None = auto, o 'cpu' / '0'
SEED = 42

RUNS_DIR = Path('./runs/optuna').resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)


def get_best_map5095(results_csv_path: Path) -> float:
    """Extrae el mejor mAP50-95(B) del CSV de resultados de Ultralytics."""
    df = pd.read_csv(results_csv_path)
    metric_col = 'metrics/mAP50-95(B)'
    if metric_col not in df.columns:
        raise KeyError(f'No se encontró la columna {metric_col} en {results_csv_path}')
    return float(df[metric_col].max())


def build_pruning_callback(trial: optuna.trial.Trial):
    """Callback para reportar mAP50-95 por época y permitir pruning real."""

    def _callback(trainer):
        metrics = getattr(trainer, 'metrics', None) or {}
        map50_95 = metrics.get('metrics/mAP50-95(B)', None)
        if map50_95 is None:
            return

        epoch = int(getattr(trainer, 'epoch', 0)) + 1
        trial.report(float(map50_95), step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned(f'Trial podado en época {epoch} con mAP50-95={map50_95:.4f}')

    return _callback


def objective(trial: optuna.trial.Trial) -> float:
    # Espacio de búsqueda (pequeño pero útil para formación)
    params = {
        'lr0': trial.suggest_float('lr0', 1e-4, 5e-2, log=True),
        'lrf': trial.suggest_float('lrf', 0.01, 0.5),
        'momentum': trial.suggest_float('momentum', 0.80, 0.98),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'hsv_h': trial.suggest_float('hsv_h', 0.0, 0.05),
        'hsv_s': trial.suggest_float('hsv_s', 0.2, 0.9),
        'hsv_v': trial.suggest_float('hsv_v', 0.2, 0.9),
        'degrees': trial.suggest_float('degrees', 0.0, 10.0),
        'scale': trial.suggest_float('scale', 0.2, 0.9),
        'fliplr': trial.suggest_float('fliplr', 0.0, 0.5),
        'mosaic': trial.suggest_float('mosaic', 0.0, 1.0),
    }

    trial_name = f'trial_{trial.number:03d}'
    trial_dir = RUNS_DIR / trial_name
    model = YOLO(MODEL_NAME)
    model.add_callback('on_fit_epoch_end', build_pruning_callback(trial))

    try:
        model.train(
            data=str(data_yaml_path),
            epochs=EPOCHS_PER_TRIAL,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            workers=WORKERS,
            device=DEVICE,
            seed=SEED,
            project=str(RUNS_DIR),
            name=trial_name,
            exist_ok=True,
            pretrained=True,
            deterministic=True,
            verbose=False,
            val=True,
            plots=True,
            **params,
        )
    except optuna.TrialPruned:
        trial.set_user_attr('trial_dir', str(trial_dir))
        raise

    results_csv = trial_dir / 'results.csv'
    best_model_path = trial_dir / 'weights' / 'best.pt'
    score = get_best_map5095(results_csv)

    # Guardamos metadatos del trial para usar luego en visualización/inferencia.
    trial.set_user_attr('best_map50_95', score)
    trial.set_user_attr('trial_dir', str(trial_dir))
    trial.set_user_attr('best_model_path', str(best_model_path))
    return score

## 3) Ejecutar Optuna
Breve: lanzamos los trials y construimos una tabla para comparar resultados.

In [ ]:
study = optuna.create_study(
    study_name='yolov8n_kortxovision_optuna',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3, interval_steps=1),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
BEST_TRIAL_NUM = best_trial.number
BEST_TRIAL_DIR = Path(best_trial.user_attrs['trial_dir'])
BEST_MODEL_PATH = Path(best_trial.user_attrs['best_model_path'])

print('Mejor trial:', BEST_TRIAL_NUM)
print('Mejor mAP50-95:', round(best_trial.value, 4))
print('Ruta best.pt:', BEST_MODEL_PATH)
print('Mejores hiperparámetros:')
for k, v in best_trial.params.items():
    print(f'  - {k}: {v}')

trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
trials_df = trials_df.sort_values('value', ascending=False).reset_index(drop=True)
trials_df.head(10)

## 5) Visualización en TensorBoard
Breve: abrimos TensorBoard para comparar los trials de entrenamiento (ideal en Colab).

In [ ]:
if 'google.colab' in sys.modules:
    ip = get_ipython()
    try:
        ip.run_line_magic('load_ext', 'tensorboard')
    except Exception:
        # Si ya está cargada la extensión, continuamos.
        pass

    print('Abriendo TensorBoard sobre:', RUNS_DIR)
    ip.run_line_magic('tensorboard', f'--logdir {RUNS_DIR}')
else:
    print('En local, lanza TensorBoard así:')
    print(f'tensorboard --logdir "{RUNS_DIR}" --port 6006')

## 6) Visualización de resultados (Optuna)
Breve: inspeccionamos convergencia, relevancia de hiperparámetros y dispersión por trial.

In [ ]:
# 6.1 Historial de optimización
ax1 = plot_optimization_history(study)
ax1.set_title('Optuna: historial de mAP50-95')
plt.show()

# 6.2 Importancia de hiperparámetros
ax2 = plot_param_importances(study)
ax2.set_title('Optuna: importancia de hiperparámetros')
plt.show()

# 6.3 Ranking simple por trial
plt.figure(figsize=(10, 4))
sns.barplot(data=trials_df.sort_values('number'), x='number', y='value', color='steelblue')
plt.title('mAP50-95 por trial')
plt.xlabel('Trial')
plt.ylabel('mAP50-95')
plt.tight_layout()
plt.show()

# 6.4 Mejor fila (tabla)
trials_df.head(1)

## 7) Cargar el mejor modelo y revisar artefactos
Breve: cargamos `best.pt` del mejor trial y mostramos curvas/matriz generadas por Ultralytics.

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))
print('Modelo cargado desde:', BEST_MODEL_PATH)

artifact_candidates = [
    BEST_TRIAL_DIR / 'results.png',
    BEST_TRIAL_DIR / 'confusion_matrix.png',
    BEST_TRIAL_DIR / 'confusion_matrix_normalized.png',
    BEST_TRIAL_DIR / 'PR_curve.png',
    BEST_TRIAL_DIR / 'P_curve.png',
    BEST_TRIAL_DIR / 'R_curve.png',
    BEST_TRIAL_DIR / 'F1_curve.png',
]

for artifact in artifact_candidates:
    if artifact.exists():
        print('Mostrando:', artifact.name)
        display(Image(filename=str(artifact)))
    else:
        print('No encontrado:', artifact.name)

## 8) Inferencia con `kortxo.jpg` usando el mejor modelo
Breve: ejecutamos predicción y visualizamos la imagen resultante con detecciones.

In [ ]:
INFER_IMAGE = Path('./kortxo.jpg').resolve()
if not INFER_IMAGE.exists():
    alt = Path('./notebooks/kortxo.jpg').resolve()
    if alt.exists():
        INFER_IMAGE = alt

if not INFER_IMAGE.exists():
    raise FileNotFoundError(
        'No encuentro kortxo.jpg. Colócalo en la raíz del repo o en notebooks/kortxo.jpg.'
    )

print('Imagen de inferencia:', INFER_IMAGE)

infer_out_dir = BEST_TRIAL_DIR / 'inference_kortxo'
infer_out_dir.mkdir(parents=True, exist_ok=True)

results = best_model.predict(
    source=str(INFER_IMAGE),
    imgsz=IMG_SIZE,
    conf=0.25,
    iou=0.7,
    save=True,
    project=str(infer_out_dir),
    name='predict',
    exist_ok=True,
    verbose=False,
)

pred_path = infer_out_dir / 'predict' / INFER_IMAGE.name
if pred_path.exists():
    display(Image(filename=str(pred_path)))
else:
    print('No se pudo localizar la imagen predicha en:', pred_path)

# Vista rápida de detecciones
if results:
    boxes = results[0].boxes
    print('Detecciones:', 0 if boxes is None else len(boxes))